In [0]:
from pyspark.sql import SparkSession, functions as F 

  

spark = SparkSession.builder.appName("day3-assignment").getOrCreate() 

  

customers = (spark.read.option("header", True) 

             .schema("""customer_id INT, first_name STRING, last_name STRING, 

                        city STRING, state STRING, signup_date DATE""") 

             .csv(("/Volumes/basicspyspark/default/learn/datasets/customers.csv"))) 

  

orders = (spark.read.option("header", True) 

          .schema("""order_id INT, order_date DATE, customer_id INT, 

                     order_status STRING, amount DOUBLE""") 

          .csv(("/Volumes/basicspyspark/default/learn/datasets/orders.csv"))) 

  

products = (spark.read.option("header", True) 

            .schema("""product_id INT, product_name STRING, 

                       category STRING, price DOUBLE""") 

            .csv("/Volumes/basicspyspark/default/learn/datasets/products.csv")) 

In [0]:
#TASK 1 Inspect a dataframe

customers.printSchema()
print(customers.columns)
print(customers.dtypes)
print(customers.count())


orders.printSchema()
print(orders.columns)
print(orders.dtypes)
print(orders.count())

products.printSchema()
print(products.columns)
print(products.dtypes)
print(products.count())

root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- signup_date: date (nullable = true)

['customer_id', 'first_name', 'last_name', 'city', 'state', 'signup_date']
[('customer_id', 'int'), ('first_name', 'string'), ('last_name', 'string'), ('city', 'string'), ('state', 'string'), ('signup_date', 'date')]
50
root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- amount: double (nullable = true)

['order_id', 'order_date', 'customer_id', 'order_status', 'amount']
[('order_id', 'int'), ('order_date', 'date'), ('customer_id', 'int'), ('order_status', 'string'), ('amount', 'double')]
300
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = tr

In [0]:

#TASK 2 select and withcolumns

# 1. Create full_name column
df = customers.withColumn(
    "full_name",
    F.concat_ws(" ", "first_name", "last_name")
)

# 2. Select required columns
df = df.select(
    "customer_id",
    "full_name",
    "city"
)

# 3. Display first 5 rows
df.show(5)


+-----------+------------+---------+
|customer_id|   full_name|     city|
+-----------+------------+---------+
|          1|  Aarav Nair|Bengaluru|
|          2|Vikram Reddy|   Mumbai|
|          3|   Varun Rao|Bengaluru|
|          4| Rohan Patel|Hyderabad|
|          5| Neha Sharma|   Mysuru|
+-----------+------------+---------+
only showing top 5 rows


In [0]:
#TASK 3 filter and orderby
result = products.filter(
    (products["category"] == "Electronics") &
    (products["price"] > 1000)
)

result = result.select("product_name", "price")

result = result.orderBy("price", ascending=False)

result.show()


+------------+-------+
|product_name|  price|
+------------+-------+
|  Headphones|4535.56|
|    Keyboard|4311.21|
|      Webcam|3376.99|
|     Monitor|1265.22|
+------------+-------+



In [0]:
#TASK 4 Distinct values
# 1. Select city and remove duplicates
cities = customers.select("city").distinct()

# 2. Count different cities
print("Total different cities:", cities.count())

# 3. Sort cities A-Z and display
cities.orderBy("city").show()


Total different cities: 10
+-------------+
|         city|
+-------------+
|    Bengaluru|
|      Chennai|
|   Coimbatore|
|        Delhi|
|    Hyderabad|
|      Kolkata|
|       Mumbai|
|       Mysuru|
|         Pune|
|Visakhapatnam|
+-------------+



In [0]:
#TASK 5 Your first Spark SQL query
# Step 1: Register customers as a temporary view
customers.createOrReplaceTempView("customers")

# Step 2: Write SQL query
result = spark.sql("""
    SELECT state, COUNT(*) AS customer_count
    FROM customers
    GROUP BY state
    ORDER BY customer_count DESC
""")

# Step 3: Display the result
result.show()


+-----+--------------+
|state|customer_count|
+-----+--------------+
|   KA|            14|
|   MH|            10|
|   TN|            10|
|   DL|             4|
|   AP|             4|
|   TG|             4|
|   WB|             4|
+-----+--------------+



In [0]:
#TASK 6

from pyspark.sql import functions as F

result_df = products.groupBy("category").agg(
    F.count("*").alias("product_count"),
    F.round(F.avg("price"), 2).alias("avg_price")
)

result_df = result_df.orderBy(
    F.col("avg_price").desc()
)

result_df.show()


+-----------+-------------+---------+
|   category|product_count|avg_price|
+-----------+-------------+---------+
| Stationery|            3|  3997.12|
|       Home|            4|  2861.62|
|Electronics|            5|  2787.04|
|      Books|            4|  2369.06|
|    Fitness|            4|  1836.77|
+-----------+-------------+---------+



In [0]:
#TASK 7
from pyspark.sql import functions as F

result_df = products.groupBy("category").agg(
    F.count("*").alias("product_count"),
    F.round(F.avg("price"), 2).alias("avg_price")
)

result_df.orderBy(
    F.col("avg_price").desc()
).show()



products.createOrReplaceTempView("products")

result_sql = spark.sql("""
    SELECT
        category,
        COUNT(*) AS product_count,
        ROUND(AVG(price), 2) AS avg_price
    FROM products
    GROUP BY category
    ORDER BY avg_price DESC
""")

result_sql.show()



+-----------+-------------+---------+
|   category|product_count|avg_price|
+-----------+-------------+---------+
| Stationery|            3|  3997.12|
|       Home|            4|  2861.62|
|Electronics|            5|  2787.04|
|      Books|            4|  2369.06|
|    Fitness|            4|  1836.77|
+-----------+-------------+---------+

+-----------+-------------+---------+
|   category|product_count|avg_price|
+-----------+-------------+---------+
| Stationery|            3|  3997.12|
|       Home|            4|  2861.62|
|Electronics|            5|  2787.04|
|      Books|            4|  2369.06|
|    Fitness|            4|  1836.77|
+-----------+-------------+---------+



In [0]:
#TASK 8
from pyspark.sql import functions as F

# 1. Inner join
joined_df = orders.join(
    customers,
    "customer_id",
    "inner"
)

# 2. Count joined rows
print("Joined row count:", joined_df.count())

# 3. Top 5 cities by order count
top_cities = joined_df.groupBy("city").agg(
    F.count("*").alias("order_count")
)

top_cities.orderBy(
    F.col("order_count").desc()
).show(5)


Joined row count: 294
+----------+-----------+
|      city|order_count|
+----------+-----------+
| Bengaluru|         49|
|    Mumbai|         41|
|   Chennai|         32|
|   Kolkata|         29|
|Coimbatore|         28|
+----------+-----------+
only showing top 5 rows


In [0]:
#TASK 9
# Find orders whose customer_id does not exist in customers
orphan_orders = orders.join(
    customers,
    on="customer_id",
    how="left_anti"
)

# Show only required columns
orphan_orders.select(
    "order_id",
    "customer_id"
).show()

# Count orphan orders
print("Orphan orders count:", orphan_orders.count())


+--------+-----------+
|order_id|customer_id|
+--------+-----------+
|      38|        999|
|      76|        999|
|      97|        999|
|     109|        999|
|     141|        999|
|     195|        999|
+--------+-----------+

Orphan orders count: 6
